# 01 · Download audio

Downloads the boat-noise-free recordings from the shared Google Drive folder into
`/scratch/$USER/capstone/audio`.

Files are fetched in batches with retries. A single bulk request times out on a folder this size.
Re-running is safe: completed files are skipped, so this cell doubles as the retry mechanism if any
file fails.

Run this once, before notebook 02.

In [ ]:
!pip install -q gdown

Paste the shared Drive **folder** link below. It must be shared as "Anyone with the link".

In [ ]:
import os, time

URL  = "https://drive.google.com/drive/folders/XXXXXXXXXXXXXXXXX"
DEST = f"/scratch/{os.environ.get('USER','username')}/capstone/audio"

BATCH_SIZE  = 20     # files per batch
FILE_SLEEP  = 1.5    # seconds between files
BATCH_SLEEP = 20     # seconds between batches
RETRIES     = 5      # attempts per file
ONLY_WAV    = True

os.makedirs(DEST, exist_ok=True)
print("saving to", os.path.basename(DEST) + "/")

In [ ]:
import gdown

AUDIO_EXTS = (".wav", ".WAV")

def enumerate_folder(url):
    """List every file in the Drive folder without downloading. Sub-folders are traversed."""
    print("listing folder...", flush=True)
    entries = gdown.download_folder(url=url, skip_download=True, quiet=False, use_cookies=False)
    if not entries:
        raise SystemExit("Could not list files. Check the link is a FOLDER shared as "
                         "'Anyone with the link'.")
    return [(e.id, os.path.basename(e.path)) for e in entries]

def download_one(file_id, out_path, retries=RETRIES):
    for attempt in range(1, retries + 1):
        try:
            got = gdown.download(id=file_id, output=out_path, quiet=True, resume=True)
            if got and os.path.exists(out_path) and os.path.getsize(out_path) > 0:
                return True
        except Exception as ex:
            print(f"    attempt {attempt}/{retries} failed: {ex}", flush=True)
        time.sleep(min(60, 3 * attempt))
    return False

If the listing stops at 50 files and the folder holds more, ask for it to be split into
sub-folders — `gdown` traverses those.

In [ ]:
files = enumerate_folder(URL)
if ONLY_WAV:
    files = [(i, n) for (i, n) in files if n.endswith(AUDIO_EXTS)]
print(f"{len(files)} files to fetch")
for _, n in files[:5]:
    print("   e.g.", n)

Safe to re-run. Files already on disk are skipped, so this retries only what failed.

In [ ]:
done, skipped, failed = 0, 0, []
n = len(files)
for start in range(0, n, BATCH_SIZE):
    batch = files[start:start + BATCH_SIZE]
    print(f"\n=== batch {start//BATCH_SIZE + 1}/{(n + BATCH_SIZE - 1)//BATCH_SIZE} "
          f"({len(batch)} files) ===", flush=True)
    for file_id, name in batch:
        out_path = os.path.join(DEST, name)
        if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
            skipped += 1; continue
        if download_one(file_id, out_path):
            done += 1
        else:
            failed.append(name); print(f"    GAVE UP on {name}", flush=True)
        time.sleep(FILE_SLEEP)
    if start + BATCH_SIZE < n:
        print(f"  pausing {BATCH_SLEEP}s...", flush=True); time.sleep(BATCH_SLEEP)

on_disk = len([f for f in os.listdir(DEST) if f.endswith(AUDIO_EXTS)])
print(f"\ndownloaded {done} | already present {skipped} | failed {len(failed)}")
for m in failed:
    print("   FAILED:", m)
print(f".wav files in {os.path.basename(DEST)}/: {on_disk}")
if failed:
    print("\nRe-run this cell to retry the failures.")

**Before running notebook 02**, confirm the recordings all share one sample rate. Full-band features
depend on it, and a mixed set would make them incomparable across recordings.

In [ ]:
import soundfile as sf, glob, collections

rates = collections.Counter(sf.info(p).samplerate
                            for p in glob.glob(os.path.join(DEST, "**", "*.wav"), recursive=True))
print("sample rates:", dict(rates))
if len(rates) > 1:
    print("WARNING: more than one sample rate. Resample to a common rate before notebook 02.")